# Esercizio 1 - Miglioramento di REINFORCE su CartPole

Questo notebook copre la prima parte del laboratorio di Deep Reinforcement Learning. L'obiettivo è partire dall'idea REINFORCE proposta nel corso, organizzare il codice in funzioni riutilizzabili e adottare un protocollo di valutazione più robusto.

Usiamo `CartPole-v1`: l'osservazione ha forma `(4,)`, lo spazio delle azioni è `Discrete(2)` e il limite dell'episodio è 500 step. La policy viene addestrata con azioni campionate e valutata separatamente con azioni greedy.

ChatGPT/Codex ha supportato la riorganizzazione del laboratorio; codice, output e interpretazioni restano sotto la responsabilità dell'autore.


> **Ambiente di esecuzione**
>
> Il laboratorio è stato eseguito in Ubuntu/WSL con il kernel `Python (DRL)`. Prima di avviare il notebook installare dalla root della repository le dipendenze e i package locali con:
>
> `python -m pip install -r requirements.txt`
>
> Gli output scientifici della run completa sono preservati. Nella configurazione di consegna sweep, training e rendering sono disattivati; tabelle e figure vengono caricate dagli artefatti versionati.


## Configurazione

Il notebook mantiene breve la sequenza sperimentale: implementazione, training, valutazione e grafici risiedono nel package installabile `dla_lab3`. Questo evita duplicazioni e consente di ispezionare e testare le funzioni fuori dal notebook.


In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

PROJECT_ROOT


PosixPath('./DLA_3')

In [4]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dla_lab3 import (
    ReinforceConfig,
    evaluate_policy,
    make_env,
    observation_scale,
    reinforce,
    set_seed,
)
from dla_lab3.paths import checkpoint_dir
from dla_lab3.plotting import plot_evaluation, plot_training_returns
from IPython.display import Image, display
from dla_lab3.policy_gradient import policy_from_env

# Sweep, training e rendering sono opt-in nella consegna.
RUN_GAMMA_SWEEP = False
RUN_REINFORCE_TRAINING = False
RUN_VISUAL_EVALUATION = False

LOAD_SAVED_RESULTS = True
DISPLAY_SAVED_RESULTS = True
USE_VERSIONED_METRICS = True


## Riferimenti e scelte metodologiche

Il flusso riprende l'implementazione REINFORCE discussa nel corso e la specifica ufficiale di `CartPole-v1`.

Le scelte operative sono:

- implementazione nel package `dla_lab3`, senza copiare funzioni lunghe nelle celle;
- azioni campionate durante il training per mantenere l'esplorazione;
- azioni greedy durante la valutazione periodica;
- return totale medio e lunghezza media dell'episodio calcolati su più episodi indipendenti.


## 1. Comprensione dell'ambiente

La cella seguente controlla spazio delle osservazioni, spazio delle azioni e alcune transizioni casuali. Gli input sono l'ambiente `CartPole-v1` e il seed `2112`; l'output atteso descrive posizione e velocità del carrello, angolo e velocità angolare del palo, reward e flag terminali.

Le azioni sono `0` (spinta a sinistra) e `1` (spinta a destra).


In [5]:
SEED = 2112
set_seed(SEED)

env = make_env("CartPole-v1", seed=SEED)
obs, info = env.reset(seed=SEED)

print("Initial observation:", obs)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Observation low:", env.observation_space.low)
print("Observation high:", env.observation_space.high)

for step in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(
        f"step={step + 1} action={action} reward={reward} "
        f"terminated={terminated} truncated={truncated}"
    )
    if terminated or truncated:
        obs, info = env.reset()

env.close()


Initial observation: [-0.04156577  0.03325528 -0.04219181  0.01717232]
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)
Observation low: [-4.8               -inf -0.41887903        -inf]
Observation high: [4.8               inf 0.41887903        inf]
step=1 action=0 reward=1.0 terminated=False truncated=False
step=2 action=0 reward=1.0 terminated=False truncated=False
step=3 action=1 reward=1.0 terminated=False truncated=False
step=4 action=1 reward=1.0 terminated=False truncated=False
step=5 action=1 reward=1.0 terminated=False truncated=False


**Interpretazione.** L'ambiente assegna reward `+1` a ogni step: un return più alto indica normalmente che il palo è rimasto in equilibrio più a lungo. Questa equivalenza vale per CartPole, ma non deve essere generalizzata ad ambienti con reward shaping differenti.


## 2. Controllo di una policy non addestrata

Prima del training istanziamo `PolicyNet` e valutiamo la policy non addestrata. Il confronto usa sia `sample`, utile per osservare l'esplorazione stocastica, sia `greedy`, cioè la modalità adottata nella valutazione periodica. Il risultato costituisce una baseline debole, non una stima della policy finale.


In [6]:
set_seed(SEED)
env = make_env("CartPole-v1", seed=SEED)
obs_scale = observation_scale("CartPole-v1")

policy = policy_from_env(env, hidden_size=128)
print(policy)

untrained_stochastic_eval = evaluate_policy(
    env,
    policy,
    obs_scale,
    episodes=20,
    mode="sample",
    seed_start=10_000,
)

untrained_greedy_eval = evaluate_policy(
    env,
    policy,
    obs_scale,
    episodes=20,
    mode="greedy",
    seed_start=11_000,
)

{
    "stochastic_untrained": untrained_stochastic_eval,
    "greedy_untrained": untrained_greedy_eval,
}


PolicyNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)


{'stochastic_untrained': {'avg_return': 22.75,
  'std_return': 10.549288749694824,
  'min_return': 9.0,
  'max_return': 50.0,
  'avg_length': 22.75,
  'returns': [16.0,
   9.0,
   17.0,
   30.0,
   13.0,
   27.0,
   23.0,
   44.0,
   15.0,
   12.0,
   38.0,
   28.0,
   50.0,
   19.0,
   26.0,
   17.0,
   19.0,
   18.0,
   16.0,
   18.0],
  'lengths': [16,
   9,
   17,
   30,
   13,
   27,
   23,
   44,
   15,
   12,
   38,
   28,
   50,
   19,
   26,
   17,
   19,
   18,
   16,
   18]},
 'greedy_untrained': {'avg_return': 9.149999618530273,
  'std_return': 0.6538348197937012,
  'min_return': 8.0,
  'max_return': 10.0,
  'avg_length': 9.149999618530273,
  'returns': [10.0,
   9.0,
   9.0,
   9.0,
   8.0,
   10.0,
   10.0,
   10.0,
   9.0,
   9.0,
   9.0,
   9.0,
   10.0,
   10.0,
   9.0,
   9.0,
   9.0,
   8.0,
   9.0,
   8.0],
  'lengths': [10,
   9,
   9,
   9,
   8,
   10,
   10,
   10,
   9,
   9,
   9,
   9,
   10,
   10,
   9,
   9,
   9,
   8,
   9,
   8]}}

## 3. Sensibilità al fattore di sconto

Lo sweep controllato confronta `gamma = 0.95`, `0.98` e `0.99` mantenendo invariati seed, rete, ottimizzatore, budget e protocollo di valutazione. Il training dello sweep è disattivato nella consegna; l'output completo resta preservato e la configurazione selezionata è `gamma=0.98`.


### Return scontato e obiettivo REINFORCE

Per un episodio di durata $T$, il return dal tempo $t$ è

$$
G_t=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.
$$

dove:

- $G_t$ è il return osservato dal passo $t$;
- $T$ è la durata della traiettoria;
- $r_{t+k}$ è la ricompensa ricevuta $k$ passi dopo $t$;
- $\gamma$ è il fattore di sconto delle ricompense future.

Il return assegna più o meno importanza alle ricompense lontane in base a $\gamma$. `compute_returns` applica lo stesso calcolo procedendo a ritroso nell'episodio; lo sweep modifica realmente `gamma` nella configurazione.

REINFORCE minimizza la media negativa del policy gradient e sottrae il bonus di entropia:

$$
\mathcal{L}_{\mathrm{REINFORCE}}
=-\frac{1}{T}\sum_t \widetilde G_t\log\pi_\theta(a_t\mid s_t)
-\beta\,\overline{\mathcal{H}}
$$

dove:

- $T$ è il numero di passi dell'episodio;
- $\widetilde G_t$ è il return grezzo oppure standardizzato;
- $\pi_\theta(a_t\mid s_t)$ è la probabilità assegnata all'azione eseguita nello stato corrente;
- $\theta$ rappresenta i parametri della policy;
- $\beta$ è il coefficiente di entropia;
- $\overline{\mathcal H}$ è l'entropia media della policy nell'episodio.

Il primo termine rinforza le azioni associate a return elevati, mentre l'entropia evita una policy prematuramente troppo concentrata. `reinforce` implementa entrambi i termini; `prepare_policy_target` può standardizzare il return usando la deviazione campionaria di `torch.std`.


In [7]:
if RUN_GAMMA_SWEEP:
    GAMMA_VALUES = [0.95, 0.98, 0.99]
    GAMMA_SWEEP_EPISODES = 500
    GAMMA_SWEEP_EVAL_EVERY = 50
    GAMMA_SWEEP_EVAL_EPISODES = 10


    def first_episode_reaching(history, threshold=500.0):
        for episode, avg_return in zip(history["eval_episodes"], history["eval_avg_returns"]):
            if avg_return >= threshold:
                return episode
        return None


    def run_gamma_experiment(gamma, seed=SEED):
        set_seed(seed)
        train_env = make_env("CartPole-v1", seed=seed)
        eval_env = make_env("CartPole-v1", seed=seed + 1000)
        obs_scale = observation_scale("CartPole-v1")

        policy_gamma = policy_from_env(train_env, hidden_size=128)
        gamma_tag = str(gamma).replace(".", "")

        config_gamma = ReinforceConfig(
            gamma=gamma,
            lr_policy=3e-4,
            num_episodes=GAMMA_SWEEP_EPISODES,
            eval_every=GAMMA_SWEEP_EVAL_EVERY,
            eval_episodes=GAMMA_SWEEP_EVAL_EPISODES,
            baseline_mode="standardize",
            entropy_coef=0.01,
            checkpoint_path=str(checkpoint_dir("gamma_sweep", f"cartpole_reinforce_gamma_{gamma_tag}.pt")),
        )

        history_gamma = reinforce(policy_gamma, train_env, eval_env, obs_scale, config_gamma)
        train_env.close()
        eval_env.close()

        return history_gamma


    gamma_histories = {}
    gamma_results = []

    for gamma in GAMMA_VALUES:
        history_gamma = run_gamma_experiment(gamma)
        gamma_histories[gamma] = history_gamma
        gamma_results.append(
            {
                "gamma": gamma,
                "best_eval_return": max(history_gamma["eval_avg_returns"]),
                "final_eval_return": history_gamma["eval_avg_returns"][-1],
                "first_episode_reaching_500": first_episode_reaching(history_gamma),
            }
        )

    gamma_results
else:
    gamma_histories = {}
    gamma_results = []
    SELECTED_GAMMA = 0.98
    print("Gamma sweep non eseguito; configurazione finale gamma=0.98.")


Gamma sweep non eseguito; configurazione finale gamma=0.98.


In [8]:
if RUN_GAMMA_SWEEP:
    plt.figure(figsize=(10, 4))

    for gamma, history_gamma in gamma_histories.items():
        plt.plot(
            history_gamma["eval_episodes"],
            history_gamma["eval_avg_returns"],
            marker="o",
            label=f"gamma={gamma}",
        )

    plt.axhline(500, linestyle="--", color="tab:green", label="Soglia CartPole")
    plt.xlabel("Episodio di training")
    plt.ylabel("Return medio della valutazione greedy")
    plt.title("CartPole-v1 - discount factor comparison")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    best_gamma_row = max(
        gamma_results,
        key=lambda row: (row["best_eval_return"], row["final_eval_return"]),
    )
    SELECTED_GAMMA = best_gamma_row["gamma"]

    best_gamma_row
else:
    print("Grafico gamma sweep disponibile nell'output eseguito.")


Grafico gamma sweep disponibile nell'output eseguito.


**Criterio di scelta.** Viene selezionato il valore di `gamma` con il miglior return di valutazione; a parità, decide il return dell'ultima valutazione. Il criterio rende esplicita la scelta, ma un singolo seed non misura tutta la variabilità dell'algoritmo.


## 4. Training REINFORCE con valutazione periodica

Il training usa azioni campionate; ogni `N=50` episodi la policy viene valutata in modalità greedy su `M=20` episodi separati. Gli output attesi sono cronologia dei return di training, return medio, lunghezza media e checkpoint migliore. Nella consegna `RUN_REINFORCE_TRAINING=False` evita di sovrascrivere il checkpoint.


In [9]:
if RUN_REINFORCE_TRAINING:
    set_seed(SEED)
    train_env = make_env("CartPole-v1", seed=SEED)
    eval_env = make_env("CartPole-v1", seed=SEED + 1000)
    obs_scale = observation_scale("CartPole-v1")

    policy = policy_from_env(train_env, hidden_size=128)

    try:
        selected_gamma = SELECTED_GAMMA
    except NameError:
        selected_gamma = 0.98

    config = ReinforceConfig(
        gamma=selected_gamma,
        lr_policy=3e-4,
        num_episodes=1000,
        eval_every=50,
        eval_episodes=20,
        baseline_mode="standardize",
        entropy_coef=0.01,
        checkpoint_path=str(checkpoint_dir("cartpole_reinforce_standardized.pt")),
    )

    history = reinforce(policy, train_env, eval_env, obs_scale, config)

    train_env.close()
    eval_env.close()
else:
    history = None
    print("Training REINFORCE non eseguito in modalita' rapida.")


Training REINFORCE non eseguito in modalita' rapida.


**Nota di riproducibilità.** La valutazione usa un ambiente separato e `torch.no_grad()`. I log del training mostrano l'avanzamento, mentre le conclusioni si basano sulle valutazioni multi-episodio e sulle curve. Il checkpoint viene scritto soltanto quando il training è attivato esplicitamente.


## 5. Analisi di training e valutazione

Il return del singolo episodio è rumoroso perché la traiettoria è campionata. La curva di valutazione periodica è più informativa: media più episodi greedy e separa l'abilità appresa dal rumore di esplorazione. Con i flag disattivati vengono mostrati CSV e figure versionati.

![Return di training REINFORCE](../figures/reinforce_training_returns.png)

![Valutazione periodica REINFORCE](../figures/reinforce_periodic_evaluation.png)


In [10]:
if history is not None:
    plot_training_returns(
        history,
        title="CartPole-v1 - REINFORCE training returns",
        solved_threshold=500,
        window=50,
    )
    plot_evaluation(
        history,
        title="CartPole-v1 - valutazione greedy REINFORCE",
        solved_threshold=500,
    )
else:
    print("Training non rieseguito: vengono mostrate le figure della run finale.")
    if DISPLAY_SAVED_RESULTS:
        display(Image(filename=PROJECT_ROOT / "figures" / "reinforce_training_returns.png"))
        display(Image(filename=PROJECT_ROOT / "figures" / "reinforce_periodic_evaluation.png"))


Curve preservate nell'output e in figures/.


In [11]:
if history is not None:
    summary = [
        {"metric": "best_eval_return", "value": history["best_eval_return"]},
        {"metric": "final_eval_return", "value": history["eval_avg_returns"][-1]},
        {"metric": "final_eval_length", "value": history["eval_avg_lengths"][-1]},
        {"metric": "checkpoint_path", "value": history["checkpoint_path"]},
    ]
    summary
else:
    display(pd.read_csv(PROJECT_ROOT / "results" / "method_summary.csv").query("method == 'REINFORCE return standardizzati'"))


,environment,method,metric,value,source
0,CartPole-v1,REINFORCE standardized returns,best_evaluation_return,489.35,notebooks/01_cartpole_reinforce_evaluation.ipy...
1,CartPole-v1,REINFORCE standardized returns,final_evaluation_return,190.20,notebooks/01_cartpole_reinforce_evaluation.ipy...


## 6. Controllo visuale qualitativo

I cinque episodi CartPole sono consolidati nel notebook [`02_cartpole_value_baseline.ipynb`](02_cartpole_value_baseline.ipynb), che usa il checkpoint value-baseline selezionato. Il rendering resta opzionale e gli HTML delle animazioni non sono incorporati.


## Conclusioni

REINFORCE standardizzato ha raggiunto un miglior return periodico di `489.35`, ma l'ultima valutazione è scesa a `190.20`. La differenza mostra l'instabilità della policy e giustifica valutazioni periodiche su più episodi anziché l'uso della sola policy finale.

Il notebook verifica l'ambiente, separa training e valutazione, confronta i fattori di sconto e registra return medio e lunghezza media. Il limite principale resta la singola run con elevata varianza; il risultato non raggiunge stabilmente il limite di 500.


## Funzioni, classi e moduli locali richiamati

La tabella è ricavata dagli import, dagli utilizzi nelle celle e dalle definizioni effettive nei package locali.

| Funzione o classe | Tipo | File di definizione | Scopo | Input principali | Output principali | Sezione |
| ----------------- | ---- | ------------------- | ----- | ---------------- | ----------------- | ------- |
| `checkpoint_dir` | Funzione | `src/dla_lab3/paths.py` | Restituisce un percorso interno a `checkpoints`. | `create: bool`, `*parts` | Percorso interno alla cartella `checkpoints`. | 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `evaluate_policy` | Funzione | `src/dla_lab3/policy_gradient.py` | Valuta una policy su più episodi. | `env`, `policy: PolicyNet`, `obs_scale: torch.Tensor`, `episodes: int`, `max_steps: int`, `mode: ActionMode`, `temperature: float`, `seed_start: int \| None` | Dizionario con media, deviazione, minimo e massimo del return, lunghezza media e array grezzi. | 2. Controllo di una policy non addestrata |
| `make_env` | Funzione | `src/dla_lab3/envs.py` | Crea un ambiente Gymnasium e ne imposta il seed. | `env_id: str`, `seed: int \| None`, `render_mode: str \| None`, `**kwargs` | Ambiente Gymnasium pronto per l'uso. | 1. Comprensione dell'ambiente; 2. Controllo di una policy non addestrata; 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `observation_scale` | Funzione | `src/dla_lab3/envs.py` | Restituisce la scala delle osservazioni per gli ambienti supportati. | `env_id: str` | Tensore float con un valore per dimensione dell'osservazione. | 2. Controllo di una policy non addestrata; 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `plot_evaluation` | Funzione | `src/dla_lab3/plotting.py` | Traccia reward e lunghezza delle valutazioni periodiche. | `history: dict`, `title: str`, `solved_threshold: float \| None` | Nessuno; mostra una figura Matplotlib senza scrivere file. | 5. Analisi di training e valutazione |
| `plot_training_returns` | Funzione | `src/dla_lab3/plotting.py` | Traccia i return grezzi per episodio e la loro media mobile. | `history: dict`, `title: str`, `solved_threshold: float \| None`, `window: int` | Nessuno; mostra una figura Matplotlib senza scrivere file. | 5. Analisi di training e valutazione |
| `policy_from_env` | Funzione | `src/dla_lab3/policy_gradient.py` | Costruisce una rete di policy da un ambiente Gymnasium. | `env`, `hidden_size: int` | `PolicyNet` configurata per l'ambiente. | 2. Controllo di una policy non addestrata; 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `PolicyNet` | Classe | `src/dla_lab3/policy_gradient.py` | Policy MLP stocastica per osservazioni continue e azioni discrete. | `obs_dim: int`, `n_actions: int`, `hidden_size: int` | Istanza configurata della classe. | Creazione della policy tramite `policy_from_env` |
| `reinforce` | Funzione | `src/dla_lab3/policy_gradient.py` | Addestra una policy con REINFORCE standard. | `policy: PolicyNet`, `env`, `eval_env`, `obs_scale: torch.Tensor`, `config: ReinforceConfig` | Dizionario con return per episodio, loss, metriche di valutazione e informazioni sul checkpoint migliore. | 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `ReinforceConfig` | Classe | `src/dla_lab3/policy_gradient.py` | Configurazione di training per gli esperimenti REINFORCE. | `gamma`, `lr_policy`, `lr_value`, `num_episodes`, `max_episode_steps`, `eval_every`, `eval_episodes`, `baseline_mode` | Istanza configurata della classe. | 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
| `run_episode` | Funzione | `src/dla_lab3/policy_gradient.py` | Esegue un episodio completo. | `env`, `policy: PolicyNet`, `obs_scale: torch.Tensor`, `max_steps: int`, `mode: ActionMode`, `temperature: float`, `seed: int \| None` | Dataclass `Episode` con tensori del rollout e statistiche di sintesi. | Interno a training e valutazione |
| `set_seed` | Funzione | `src/dla_lab3/seed.py` | Imposta i seed di Python, NumPy e PyTorch. | `seed: int` | Nessuno. Modifica lo stato globale dei generatori casuali e, se disponibile, quello di tutti i device CUDA. | 1. Comprensione dell'ambiente; 2. Controllo di una policy non addestrata; 3. Sensibilità al fattore di sconto; 4. Training REINFORCE con valutazione periodica |
